# Radiomics preprocessing and feature selection


In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
while not (PROJECT_ROOT / "modeling_pipeline.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Run this code from the MMDLPC_Code_PDF folder.")
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT.parent / "MMDLPC_Code_PDF_Data"
OUTPUT_DIR = PROJECT_ROOT / '02_Radiomics/00_Preprocessing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_INPUT_DIR = Path(os.environ.get('MMDLPC_EXTERNAL_INPUTS', str(DATA_ROOT / 'External_Inputs')))

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
sys.path.insert(0, str(PROJECT_ROOT))
from modeling_pipeline import (
    read_indexed, parse_p_value, correlation_keep_indices, make_pipeline,
    training_cv, best_finite_parameters, fit_on_training,
    model_estimators, training_search_grids, positive_probability,
)
DATA_DIR = DATA_ROOT / '00_Shared_Data_and_Code/Data'


## Radiomics


In [ ]:
import os
from IPython.display import display
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

os.makedirs(str(OUTPUT_DIR), exist_ok=True)
os.makedirs(str(OUTPUT_DIR), exist_ok=True)
os.makedirs(str(OUTPUT_DIR), exist_ok=True)
# 设置数据目录
# mydir = r'你自己数据的路径'
mydir = str(EXTERNAL_INPUT_DIR / "MRI_Images_and_Masks")
# 对应的标签文件
# labelf = r'你自己标注数据的文件地址'
labelf = str(DATA_ROOT / '00_Shared_Data_and_Code/Data/CPGEA-TCGA 20230106 OK.csv')
# 读取标签数据列名
labels = ['HRR_ANY']
# 前缀
prefix = 'Rad_'


In [ ]:
if not (DATA_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/rad_features_reference.csv').exists():
    from pathlib import Path
    from onekey_algo.custom.components.Radiology import diagnose_3d_image_mask_settings, get_image_mask_from_dir
    
    # 生成images和masks对，一对一的关系。也可以自定义替换。
    images, masks = get_image_mask_from_dir(mydir, images='images', masks='masks')
    
    # 自定义获取images和masks数据的方法，下面的例子为，每个样本一个文件夹，图像是以im.nii结尾，mask是以seg.nii结尾。
    # def get_images_mask(mydir):
    #     images = []
    #     masks = []
    #     for root, dirs, files in os.walk(mydir):
    #         for f in files:
    #             if f.endswith('im.nii'):
    #                 images.append(os.path.join(root, f))
    #             if f.endswith('seg.nii'):
    #                 masks.append(os.path.join(root, f))
    #     return images, masks
    # images, masks = get_images_mask(mydir)
    
    # diagnose_3d_image_mask_settings(images, masks, verbose=True)
    print(f'获取到{len(images)}个样本。')


In [ ]:
import pandas as pd
 


if os.path.exists(str(DATA_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/rad_features_reference.csv')):
    rad_data = pd.read_csv(str(DATA_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/rad_features_reference.csv'), header=0)
else:
    from onekey_algo.custom.components.Radiology import ConventionalRadiomics
    # 如果要自定义一些特征提取方式，可以使用param_file。
    param_file = str(PROJECT_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/radiomics_extraction_MR_3mm.yaml')
#     param_file = None
    radiomics = ConventionalRadiomics(param_file, correctMask=True)
    radiomics.extract(images, masks)
    rad_data = radiomics.get_label_data_frame(label=1)
    rad_data.columns = [c.replace('-', '_') for c in rad_data.columns]
    rad_data.to_csv(str(OUTPUT_DIR / 'Radiomics_rad_features.csv'), header=True, index=False)
rad_data


In [ ]:
series = []
for ending, suffix in [('-1.nii.gz', 'DWI'), ('-2.nii.gz', 'T2WI')]:
    frame = rad_data.loc[rad_data['ID'].str.endswith(ending)].copy()
    frame['ID'] = frame['ID'].str.removesuffix(ending)
    if frame.ID.isna().any() or frame.ID.duplicated().any():
        raise ValueError('Duplicate or missing patient/sequence IDs.')
    series.append(frame.set_index('ID').add_suffix(suffix))
rad_data = series[0].join(series[1], how='inner', validate='one_to_one').reset_index()


In [ ]:
import matplotlib.pyplot as plt
sorted_counts = pd.DataFrame([c.split('_')[-2] for c in rad_data.columns if c !='ID']).value_counts()
plt.pie(sorted_counts, labels=[i[0] for i in sorted_counts.index], startangle=0,
        counterclock = False, autopct = '%.1f%%')
display(sorted_counts)
plt.savefig(str(OUTPUT_DIR / f'Reference_{prefix}feature_ratio.svg'), bbox_inches = 'tight')
plt.savefig(str(OUTPUT_DIR / f'Reference_{prefix}feature_ratio.pdf'), bbox_inches = 'tight')


In [ ]:
radiology_features = rad_data.set_index('ID')


In [ ]:
partition = read_indexed(DATA_DIR / 'R_fixed_partition.csv')
if set(partition.Split) != {'Train', 'Test'}:
    raise ValueError('Expected a pre-specified Train/Test partition.')
label_table = read_indexed(DATA_DIR / 'CPGEA-TCGA 20230106 OK.csv')
if not partition.index.isin(radiology_features.index).all() or not partition.index.isin(label_table.index).all():
    raise ValueError('The feature or label table is missing patients in the partition.')
raw_features = radiology_features.loc[partition.index].apply(pd.to_numeric, errors='raise')
raw_features = raw_features.replace([np.inf, -np.inf], np.nan)
X_source = raw_features.add_prefix('radiology__')
y_source = label_table.loc[partition.index, 'HRR_ANY']
if not y_source.isin([0, 1]).all():
    raise ValueError('Expected observed binary HRR_ANY labels.')
y_source = y_source.astype(int)
split = partition.Split
train_rows = split.eq('Train')
test_rows = split.eq('Test')
train_ids = X_source.index[train_rows]
test_ids = X_source.index[test_rows]
if not X_source.index.is_unique or not set(train_ids).isdisjoint(test_ids):
    raise ValueError('Patient IDs must be unique and the partition must be disjoint.')
labels = ['HRR_ANY']
label_data = label_table.loc[partition.index, labels + ['group']].reset_index()
ids = pd.Series(partition.index, index=partition.index, name='ID')

combined_data = raw_features.join(label_table.loc[partition.index, labels + ['group']], validate='one_to_one')


In [ ]:
combined_data.describe()


In [ ]:
observed_columns = raw_features.columns[raw_features.loc[train_rows].notna().any()]
imputer = SimpleImputer(strategy='median').fit(raw_features.loc[train_rows, observed_columns])
filled = imputer.transform(raw_features[observed_columns])
scaler = StandardScaler().fit(filled[train_rows.to_numpy()])
data = pd.DataFrame(scaler.transform(filled), index=partition.index, columns=observed_columns)
data = data.join(label_table.loc[partition.index, labels + ['group']], validate='one_to_one')
data.describe()


In [ ]:
feature_columns = [column for column in data if column not in labels + ['group']]


In [ ]:
import seaborn as sns
from scipy.stats import ttest_ind
training_features = data.loc[train_rows, feature_columns]
training_labels = y_source.loc[train_rows]
feature_tests = ttest_ind(training_features.loc[training_labels.eq(0)],
                         training_features.loc[training_labels.eq(1)], axis=0, equal_var=False)
stats = pd.DataFrame({'feature_name': feature_columns, 'pvalue': feature_tests.pvalue})
stats


In [ ]:
import matplotlib.pyplot as plt

def map2float(x):
    return parse_p_value(x)

stats[['pvalue']] = stats[['pvalue']].applymap(map2float)
stats[['group']] = stats[['feature_name']].applymap(lambda x: x.split('_')[-2])
stats = stats[['feature_name', 'pvalue', 'group']]
g = sns.catplot(x="group", y="pvalue", data=stats, kind="violin")
g.fig.set_size_inches(15,10)
sns.stripplot(x="group", y="pvalue", data=stats, ax=g.ax, color='black')
plt.savefig(str(OUTPUT_DIR / f'Reference_{prefix}feature_stats.svg'), bbox_inches = 'tight')
plt.savefig(str(OUTPUT_DIR / f'Reference_{prefix}feature_stats.pdf'), bbox_inches = 'tight')


In [ ]:
pvalue = 0.05
sel_feature = stats.loc[stats.pvalue.lt(pvalue), 'feature_name'].tolist()
if not sel_feature:
    raise ValueError('No features passed training-only statistical screening.')


In [ ]:
pearson_corr = data.loc[train_rows, sel_feature].corr('pearson')


In [ ]:
pearson_corr


In [ ]:
positions = correlation_keep_indices(data.loc[train_rows, sel_feature].to_numpy(), 'pearson', .9)
sel_feature = [sel_feature[i] for i in positions]


In [ ]:
sel_data = data[sel_feature + labels + ['group']]
sel_data.describe()


In [ ]:
stats.loc[stats.feature_name.isin(sel_feature)].sort_values('pvalue')


In [ ]:
X_data = X_source.loc[train_rows].copy()
X_test_data = X_source.loc[test_rows].copy()
y_data = y_source.loc[train_rows].to_frame('HRR_ANY')
y_test_data = y_source.loc[test_rows].to_frame('HRR_ANY')
n_classes = 2


In [ ]:
selection_classifier = LogisticRegression(max_iter=1000, random_state=0)
selection_grid = training_search_grids('R', {'LR': selection_classifier})['LR']
selection_grid['features__R__pvalue_threshold'] = [0.05]
selection_search = GridSearchCV(
    make_pipeline(X_data, 'R', selection_classifier), selection_grid,
    scoring='roc_auc', cv=training_cv(y_data['HRR_ANY']),
    n_jobs=1, error_score=np.nan, refit=False)
selection_search.fit(X_data, y_data['HRR_ANY'])
selection_parameters = best_finite_parameters(selection_search)
selection_pipeline = make_pipeline(X_data, 'R', selection_classifier)
selection_pipeline.set_params(**selection_parameters).fit(X_data, y_data['HRR_ANY'])
selector = selection_pipeline['features'].named_transformers_['R']
alpha = selector.alpha


In [ ]:
selection_results = pd.DataFrame(selection_search.cv_results_)
alpha_key = next(key for key in selection_results if key.endswith('__alpha'))
plt.figure(figsize=(6, 4))
plt.errorbar(selection_results[alpha_key].astype(float), selection_results['mean_test_score'],
             yerr=selection_results['std_test_score'], marker='o', markersize=3)
plt.xscale('log')
plt.xlabel('LASSO alpha')
plt.ylabel('Training cross-validation AUC (mean and SD)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'Reference_Rad_feature_selection_CV_AUC.pdf', bbox_inches='tight')


In [ ]:
models = [selector.lasso_]
column_names = np.asarray(selector.observed_columns_)[selector.correlation_indices_]


In [ ]:
selected_features = [selector.selected_features_.tolist()]
feat_coef = [(name, coefficient) for name, coefficient in zip(column_names, selector.lasso_.coef_)
             if abs(coefficient) > 1e-6]
feat_coef_df = pd.DataFrame(feat_coef, columns=['feature_name', 'Coefficients'])
feat_coef_df


In [ ]:
feat_coef = sorted(feat_coef, key=lambda x: x[1])
feat_coef_df = pd.DataFrame(feat_coef, columns=['feature_name', 'Coefficients'])
feat_coef_df.plot(x='feature_name', y='Coefficients', kind='barh')

plt.savefig(str(OUTPUT_DIR / f'Reference_{prefix}feature_weights.svg'), bbox_inches = 'tight')
plt.savefig(str(OUTPUT_DIR / f'Reference_{prefix}feature_weights.pdf'), bbox_inches = 'tight')


In [ ]:
selected_values = pd.DataFrame(selection_pipeline['features'].transform(X_source),
    index=X_source.index, columns=selection_pipeline['features'].get_feature_names_out())
selected_values.index.name = 'ID'
selected_values.reset_index().to_csv(OUTPUT_DIR / 'rad_sel_features_reference.csv', index=False)
selected_values.columns
